In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk
import webbrowser
import os

In [2]:
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\aritr\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [3]:
apps_df=pd.read_csv('Play Store Data.csv')
reviews_df=pd.read_csv('User Reviews.csv')

In [4]:
apps_df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


In [5]:
reviews_df.head()

,App,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity
0,10 Best Foods for You,I like eat delicious food. That's I'm cooking ...,Positive,1.00,0.533333
1,10 Best Foods for You,This help eating healthy exercise regular basis,Positive,0.25,0.288462
2,10 Best Foods for You,NaN,NaN,NaN,NaN
3,10 Best Foods for You,Works great especially going grocery store,Positive,0.40,0.875000
4,10 Best Foods for You,Best idea us,Positive,1.00,0.300000


In [6]:
apps_df = apps_df.dropna(subset=['Rating'])
for column in apps_df.columns :
    apps_df[column].fillna(apps_df[column].mode()[0],inplace=True)
apps_df.drop_duplicates(inplace=True)
apps_df=apps_df=apps_df[apps_df['Rating']<=5]
reviews_df.dropna(subset=['Translated_Review'],inplace=True)

C:\Users\aritr\AppData\Local\Temp\ipykernel_18412\1038863187.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  apps_df[column].fillna(apps_df[column].mode()[0],inplace=True)


In [7]:
apps_df.dtypes

App                object
Category           object
Rating            float64
Reviews            object
Size               object
Installs           object
Type               object
Price              object
Content Rating     object
Genres             object
Last Updated       object
Current Ver        object
Android Ver        object
dtype: object

In [8]:
#Convert the Installs columns to numeric by removing commas and +
apps_df['Installs']=apps_df['Installs'].str.replace(',','').str.replace('+','').astype(int)

#Convert Price column to numeric after removing $
apps_df['Price']=apps_df['Price'].str.replace('$','').astype(float)

In [9]:
apps_df.dtypes

App                object
Category           object
Rating            float64
Reviews            object
Size               object
Installs            int64
Type               object
Price             float64
Content Rating     object
Genres             object
Last Updated       object
Current Ver        object
Android Ver        object
dtype: object

In [10]:
merged_df=pd.merge(apps_df,reviews_df,on='App',how='inner')

In [11]:
merged_df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity
0,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,A kid's excessive ads. The types ads allowed a...,Negative,-0.250,1.000000
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,It bad >:(,Negative,-0.725,0.833333
2,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,like,Neutral,0.000,0.000000
3,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,I love colors inspyering,Positive,0.500,0.600000
4,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,I hate,Negative,-0.800,0.900000


In [12]:
def convert_size(size):
    if 'M' in size:
        return float(size.replace('M',''))
    elif 'k' in size:
        return float(size.replace('k',''))/1024
    else:
        return np.nan
apps_df['Size']=apps_df['Size'].apply(convert_size)

In [13]:
apps_df

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19.0,10000,Free,0.0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14.0,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7,5000000,Free,0.0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25.0,50000000,Free,0.0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8,100000,Free,0.0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up
...,...,...,...,...,...,...,...,...,...,...,...,...,...
10834,FR Calculator,FAMILY,4.0,7,2.6,500,Free,0.0,Everyone,Education,"June 18, 2017",1.0.0,4.1 and up
10836,Sya9a Maroc - FR,FAMILY,4.5,38,53.0,5000,Free,0.0,Everyone,Education,"July 25, 2017",1.48,4.1 and up
10837,Fr. Mike Schmitz Audio Teachings,FAMILY,5.0,4,3.6,100,Free,0.0,Everyone,Education,"July 6, 2018",1.0,4.1 and up
10839,The SCP Foundation DB fr nn5n,BOOKS_AND_REFERENCE,4.5,114,NaN,1000,Free,0.0,Mature 17+,Books & Reference,"January 19, 2015",Varies with device,Varies with device


In [14]:
#Lograrithmic
apps_df['Log_Installs']=np.log(apps_df['Installs'])

In [15]:
apps_df['Reviews']=apps_df['Reviews'].astype(int)

In [16]:
apps_df['Log_Reviews']=np.log(apps_df['Reviews'])

In [17]:
apps_df.dtypes

App                object
Category           object
Rating            float64
Reviews             int64
Size              float64
Installs            int64
Type               object
Price             float64
Content Rating     object
Genres             object
Last Updated       object
Current Ver        object
Android Ver        object
Log_Installs      float64
Log_Reviews       float64
dtype: object

In [18]:
def rating_group(rating):
    if rating >= 4:
        return 'Top rated app'
    elif rating >=3:
        return 'Above average'
    elif rating >=2:
        return 'Average'
    else:
        return 'Below Average'
apps_df['Rating_Group']=apps_df['Rating'].apply(rating_group)

In [19]:
#Revenue column
apps_df['Revenue']=apps_df['Price']*apps_df['Installs']

In [20]:
sia = SentimentIntensityAnalyzer()

In [21]:
review = "This app is amazing! I love the new features."
sentiment_score= sia.polarity_scores(review)
print(sentiment_score)

{'neg': 0.0, 'neu': 0.42, 'pos': 0.58, 'compound': 0.8516}


In [22]:
reviews_df['Sentiment_Score']=reviews_df['Translated_Review'].apply(lambda x: sia.polarity_scores(str(x))['compound'])

In [23]:
reviews_df.head()

,App,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity,Sentiment_Score
0,10 Best Foods for You,I like eat delicious food. That's I'm cooking ...,Positive,1.00,0.533333,0.9531
1,10 Best Foods for You,This help eating healthy exercise regular basis,Positive,0.25,0.288462,0.6597
3,10 Best Foods for You,Works great especially going grocery store,Positive,0.40,0.875000,0.6249
4,10 Best Foods for You,Best idea us,Positive,1.00,0.300000,0.6369
5,10 Best Foods for You,Best way,Positive,1.00,0.300000,0.6369


In [24]:
apps_df['Last Updated']=pd.to_datetime(apps_df['Last Updated'],errors='coerce')

In [25]:
apps_df['Year']=apps_df['Last Updated'].dt.year

In [26]:
apps_df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Log_Installs,Log_Reviews,Rating_Group,Revenue,Year
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19.0,10000,Free,0.0,Everyone,Art & Design,2018-01-07,1.0.0,4.0.3 and up,9.210340,5.068904,Top rated app,0.0,2018
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14.0,500000,Free,0.0,Everyone,Art & Design;Pretend Play,2018-01-15,2.0.0,4.0.3 and up,13.122363,6.874198,Above average,0.0,2018
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7,5000000,Free,0.0,Everyone,Art & Design,2018-08-01,1.2.4,4.0.3 and up,15.424948,11.379508,Top rated app,0.0,2018
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25.0,50000000,Free,0.0,Teen,Art & Design,2018-06-08,Varies with device,4.2 and up,17.727534,12.281384,Top rated app,0.0,2018
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8,100000,Free,0.0,Everyone,Art & Design;Creativity,2018-06-20,1.1,4.4 and up,11.512925,6.874198,Top rated app,0.0,2018


In [27]:
html_files_path="./"
if not os.path.exists(html_files_path):
    os.makedirs(html_files_path)

In [28]:
plot_containers=""

In [29]:
# Save each Plotly figure to an HTML file
def save_plot_as_html(fig, filename, insight):
    global plot_containers
    filepath = os.path.join(html_files_path, filename)
    html_content = pio.to_html(fig, full_html=False, include_plotlyjs='inline')
    # Append the plot and its insight to plot_containers
    plot_containers += f"""
    <div class="plot-container" id="{filename}" onclick="openPlot('{filename}')">
        <div class="plot">{html_content}</div>
        <div class="insights">{insight}</div>
    </div>
    """
    fig.write_html(filepath, full_html=False, include_plotlyjs='inline')

In [30]:
plot_width=400
plot_height=300
plot_bg_color='black'
text_color='white'
title_font={'size':16}
axis_font={'size':12}

In [31]:
#Figure 1
category_counts=apps_df['Category'].value_counts().nlargest(10)
fig1=px.bar(
    x=category_counts.index,
    y=category_counts.values,
    labels={'x':'Category','y':'Count'},
    title='Top Categories on Play Store',
    color=category_counts.index,
    color_discrete_sequence=px.colors.sequential.Plasma,
    width=400,
    height=300
)
fig1.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size':16},
    xaxis=dict(title_font={'size':12}),
    yaxis=dict(title_font={'size':12}),
    margin=dict(l=10,r=10,t=30,b=10)
)
save_plot_as_html(fig1,"Category Graph 1.html","The top categories on the Play Store are dominated by tools, entertainment, and productivity apps")
            

In [32]:
#Figure 2
type_counts=apps_df['Type'].value_counts()
fig2=px.pie(
    values=type_counts.values,
    names=type_counts.index,
    title='App Type Distribution',
    color_discrete_sequence=px.colors.sequential.RdBu,
    width=400,
    height=300
)
fig2.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size':16},
    margin=dict(l=10,r=10,t=30,b=10)
)
save_plot_as_html(fig2,"Type Graph 2.html","Most apps on the Playstore are free, indicating a strategy to attract users first and monetize through ads or in app purchases")

In [33]:
#Figure 3
fig3=px.histogram(
    apps_df,
    x='Rating',
    nbins=20,
    title='Rating Distribution',
    color_discrete_sequence=['#636EFA'],
    width=400,
    height=300
)
fig3.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size':16},
    xaxis=dict(title_font={'size':12}),
    yaxis=dict(title_font={'size':12}),
    margin=dict(l=10,r=10,t=30,b=10)
)
save_plot_as_html(fig3,"Rating Graph 3.html","Ratings are skewed towards higher values, suggesting that most apps are rated favorably by users")

In [34]:
#Figure 4
sentiment_counts=reviews_df['Sentiment_Score'].value_counts()
fig4=px.bar(
    x=sentiment_counts.index,
    y=sentiment_counts.values,
    labels={'x':'Sentiment Score','y':'Count'},
    title='Sentiment Distribution',
    color=sentiment_counts.index,
    color_discrete_sequence=px.colors.sequential.RdPu,
    width=400,
    height=300
)
fig4.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size':16},
    xaxis=dict(title_font={'size':12}),
    yaxis=dict(title_font={'size':12}),
    margin=dict(l=10,r=10,t=30,b=10)
)
save_plot_as_html(fig4,"Sentiment Graph 4.html","Sentiments in reviews show a mix of positive and negative feedback, with a slight lean towards positive sentiments")

In [35]:
#Figure 5
installs_by_category=apps_df.groupby('Category')['Installs'].sum().nlargest(10)
fig5=px.bar(
    x=installs_by_category.index,
    y=installs_by_category.values,
    orientation='h',
    labels={'x':'Installs','y':'Category'},
    title='Installs by Category',
    color=installs_by_category.index,
    color_discrete_sequence=px.colors.sequential.Blues,
    width=400,
    height=300
)
fig5.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size':16},
    xaxis=dict(title_font={'size':12}),
    yaxis=dict(title_font={'size':12}),
    margin=dict(l=10,r=10,t=30,b=10)
)

save_plot_as_html(fig5,"Installs Graph 5.html","The categories with the most installs are social and communication apps, reflecting their broad appeal and daily usage")

In [36]:
# Updates Per Year Plot
updates_per_year = apps_df['Last Updated'].dt.year.value_counts().sort_index()
fig6 = px.line(
    x=updates_per_year.index,
    y=updates_per_year.values,
    labels={'x': 'Year', 'y': 'Number of Updates'},
    title='Number of Updates Over the Years',
    color_discrete_sequence=['#AB63FA'],
    width=plot_width,
    height=plot_height
)
fig6.update_layout(
    plot_bgcolor=plot_bg_color,
    paper_bgcolor=plot_bg_color,
    font_color=text_color,
    title_font=title_font,
    xaxis=dict(title_font=axis_font),
    yaxis=dict(title_font=axis_font),
    margin=dict(l=10, r=10, t=30, b=10)
)
save_plot_as_html(fig6, "Updates Graph 6.html", "Updates have been increasing over the years, showing that developers are actively maintaining and improving their apps.")

In [37]:
#Figure 7
revenue_by_category=apps_df.groupby('Category')['Revenue'].sum().nlargest(10)
fig7=px.bar(
    x=installs_by_category.index,
    y=installs_by_category.values,
    labels={'x':'Category','y':'Revenue'},
    title='Revenue by Category',
    color=installs_by_category.index,
    color_discrete_sequence=px.colors.sequential.Greens,
    width=400,
    height=300
)
fig7.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size':16},
    xaxis=dict(title_font={'size':12}),
    yaxis=dict(title_font={'size':12}),
    margin=dict(l=10,r=10,t=30,b=10)
)
save_plot_as_html(fig7,"Revenue Graph 7.html","Categories such as Business and Productivity lead in revenue generation, indicating their monetization potential")

In [38]:
#Figure 8
genre_counts=apps_df['Genres'].str.split(';',expand=True).stack().value_counts().nlargest(10)
fig8=px.bar(
    x=genre_counts.index,
    y=genre_counts.values,
    labels={'x':'Genre','y':'Count'},
    title='Top Genres',
    color=installs_by_category.index,
    color_discrete_sequence=px.colors.sequential.OrRd,
    width=400,
    height=300
)
fig8.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size':16},
    xaxis=dict(title_font={'size':12}),
    yaxis=dict(title_font={'size':12}),
    margin=dict(l=10,r=10,t=30,b=10)
)
save_plot_as_html(fig8,"Genre Graph 8.html","Action and Casual genres are the most common, reflecting users' preference for engaging and easy-to-play games")

In [39]:
#Figure 9
fig9=px.scatter(
    apps_df,
    x='Last Updated',
    y='Rating',
    color='Type',
    title='Impact of Last Update on Rating',
    color_discrete_sequence=px.colors.qualitative.Vivid,
    width=400,
    height=300
)
fig9.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size':16},
    xaxis=dict(title_font={'size':12}),
    yaxis=dict(title_font={'size':12}),
    margin=dict(l=10,r=10,t=30,b=10)
)
save_plot_as_html(fig9,"Update Graph 9.html","The Scatter Plot shows a weak correlation between the last update and ratings, suggesting that more frequent updates dont always result in better ratings.")

In [40]:
#Figure 10
fig10=px.box(
    apps_df,
    x='Type',
    y='Rating',
    color='Type',
    title='Rating for Paid vs Free Apps',
    color_discrete_sequence=px.colors.qualitative.Pastel,
    width=400,
    height=300
)
fig10.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size':16},
    xaxis=dict(title_font={'size':12}),
    yaxis=dict(title_font={'size':12}),
    margin=dict(l=10,r=10,t=30,b=10)
)

save_plot_as_html(fig10,"Paid Free Graph 10.html","Paid apps generally have higher ratings compared to free apps, suggesting that users expect higher quality from apps they pay for")

In [41]:
from datetime import datetime
import pytz
import re
from datetime import datetime, timezone, timedelta

In [46]:
#Question 1
filtered_df = apps_df[
    (apps_df['Rating'] >= 4.0) &
    (apps_df['Size'] >= 10) &
    (apps_df['Last Updated'].dt.month == 1)
]

top_categories = (
    filtered_df.groupby('Category')['Installs']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .index
)

top_df = filtered_df[filtered_df['Category'].isin(top_categories)]


result = top_df.groupby('Category').agg(Avg_Rating=('Rating','mean'),Total_Reviews=('Reviews','sum')
).reset_index()

# Convert to long format for grouped bars
plot_df = result.melt(id_vars='Category',
                      value_vars=['Avg_Rating','Total_Reviews'],
                      var_name='Metric',
                      value_name='Value')


TimeWindow = timezone(timedelta(hours=5, minutes=30))
current_time = datetime.now(TimeWindow)

if 15 <= current_time.hour < 17:

    fig11 = px.bar(
        plot_df,
        x='Category',
        y='Value',
        color='Metric',
        barmode='group',
        title='Average Rating vs Total Reviews (Top Categories)',
        color_discrete_sequence=px.colors.qualitative.Pastel,
        width=400,
        height=300
    )

    fig11.update_layout(
        plot_bgcolor='black',
        paper_bgcolor='black',
        font_color='white',
        title_font={'size':16},
        xaxis=dict(title_font={'size':12}),
        yaxis=dict(title_font={'size':12}),
        margin=dict(l=10,r=10,t=30,b=10)
    )

    save_plot_as_html(fig11,"Rating Reviews Graph 11.html","Comparison of average rating and total reviews for top app categories based on installs")



In [47]:
apps_df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Log_Installs,Log_Reviews,Rating_Group,Revenue,Year
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19.0,10000,Free,0.0,Everyone,Art & Design,2018-01-07,1.0.0,4.0.3 and up,9.210340,5.068904,Top rated app,0.0,2018
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14.0,500000,Free,0.0,Everyone,Art & Design;Pretend Play,2018-01-15,2.0.0,4.0.3 and up,13.122363,6.874198,Above average,0.0,2018
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7,5000000,Free,0.0,Everyone,Art & Design,2018-08-01,1.2.4,4.0.3 and up,15.424948,11.379508,Top rated app,0.0,2018
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25.0,50000000,Free,0.0,Teen,Art & Design,2018-06-08,Varies with device,4.2 and up,17.727534,12.281384,Top rated app,0.0,2018
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8,100000,Free,0.0,Everyone,Art & Design;Creativity,2018-06-20,1.1,4.4 and up,11.512925,6.874198,Top rated app,0.0,2018


In [48]:
#Question 2
apps_df['Android_Ver'] = apps_df['Android Ver'].str.extract('(\d+\.\d+)').astype(float)

apps_df['Revenue'] = apps_df['Installs'] * apps_df['Price']

filtered_df = apps_df[
    (apps_df['Installs'] >= 10000) &
    (apps_df['Revenue'] >= 10000) &
    (apps_df['Android_Ver'] > 4.0) &
    (apps_df['Size'] > 15) &
    (apps_df['Content Rating'] == 'Everyone') &
    (apps_df['App'].str.len() <= 30)
]


top_categories = (
    filtered_df.groupby('Category')['Installs']
    .sum()
    .sort_values(ascending=False)
    .head(3)
    .index
)

top_df = filtered_df[filtered_df['Category'].isin(top_categories)]


result = top_df.groupby(['Category','Type']).agg(
    Avg_Installs=('Installs','mean'),
    Avg_Revenue=('Revenue','mean')
).reset_index()


plot_df = result.melt(
    id_vars=['Category','Type'],
    value_vars=['Avg_Installs','Avg_Revenue'],
    var_name='Metric',
    value_name='Value'
)

TimeWindow = timezone(timedelta(hours=5, minutes=30))
current_time = datetime.now(TimeWindow)

if 13 <= current_time.hour < 14:

    fig12 = px.bar(
        plot_df,
        x='Category',
        y='Value',
        color='Metric',
        barmode='group',
        facet_col='Type',
        title='Average Installs vs Average Revenue (Free vs Paid)',
        color_discrete_sequence=px.colors.qualitative.Pastel,
        width=500,
        height=350
    )

    fig12.update_layout(
        plot_bgcolor='black',
        paper_bgcolor='black',
        font_color='white',
        title_font={'size':16},
        xaxis=dict(title_font={'size':12}),
        yaxis=dict(title_font={'size':12}),
        margin=dict(l=10,r=10,t=30,b=10)
    )

    save_plot_as_html(fig12,"Installs_Revenue_Graph.html","Dual axis comparison of average installs and revenue for free vs paid apps in top categories")


In [49]:
# Question 3

filtered_df = apps_df[(~apps_df['Category'].str.startswith(('A','C','G','S')))]

top_categories = (
    filtered_df.groupby('Category')['Installs']
    .sum()
    .sort_values(ascending=False)
    .head(5)
    .index
)

top_df = filtered_df[filtered_df['Category'].isin(top_categories)]


result = top_df.groupby('Category').agg(
    Total_Installs=('Installs','sum')
).reset_index()


result['Highlight'] = result['Total_Installs'].apply(
    lambda x: 'Above 1M' if x > 1000000 else 'Below 1M'
)


TimeWindow = timezone(timedelta(hours=5, minutes=30))
current_time = datetime.now(TimeWindow)

if 18 <= current_time.hour < 20:

    fig13 = px.choropleth(
        result,
        locations='Category',
        color='Total_Installs',
        title='Global Installs by App Category',
        color_continuous_scale='Viridis'
    )

    fig13.update_layout(
        plot_bgcolor='black',
        paper_bgcolor='black',
        font_color='white'
    )

    save_plot_as_html(fig13,"Global_Installs_Map.html","Installs visualization for top app categories")

In [50]:
#Question 4
filtered_df = apps_df[
    (apps_df['Rating'] >= 4.2) &
    (apps_df['Reviews'] > 1000) &
    (apps_df['Category'].str.startswith(('T','P'))) &
    (apps_df['App'].str.contains(r'\d', regex=True) == False) &
    (apps_df['Size'] >= 20) &
    (apps_df['Size'] <= 80)
]


filtered_df['Last Updated'] = pd.to_datetime(filtered_df['Last Updated'], errors='coerce')
filtered_df['Month'] = filtered_df['Last Updated'].dt.to_period('M').astype(str)


result = filtered_df.groupby(['Month','Category']).agg(Total_Installs=('Installs','sum')).reset_index()

result['MoM_Growth'] = result.groupby('Category')['Total_Installs'].pct_change()

result['Highlight'] = result['MoM_Growth'].apply(
    lambda x: 'High Growth' if x > 0.25 else 'Normal'
)


translation = {
    'Travel & Local': 'Voyage et Local',   
    'Productivity': 'Productividad',      
    'Photography': '写真'                  
}

result['Category_Translated'] = result['Category'].replace(translation)


TimeWindow = timezone(timedelta(hours=5, minutes=30))
current_time = datetime.now(TimeWindow)



if 16 <= current_time.hour < 18:

    fig14 = px.area(
        result,
        x='Month',
        y='Total_Installs',
        color='Category_Translated',
        title='Cumulative Installs Over Time by Category',
        line_group='Category_Translated'
    )

    fig14.update_layout(
        plot_bgcolor='black',
        paper_bgcolor='black',
        font_color='white',
        title_font={'size':16},
        margin=dict(l=10,r=10,t=30,b=10)
    )

    save_plot_as_html(fig14,"Stacked_Area_Installs.html","Stacked area chart showing cumulative installs by category over time")

C:\Users\aritr\AppData\Local\Temp\ipykernel_18412\2954668801.py:12: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\aritr\AppData\Local\Temp\ipykernel_18412\2954668801.py:13: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [51]:
filtered_df

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Log_Installs,Log_Reviews,Rating_Group,Revenue,Year,Android_Ver,Month
2802,"Shutterfly: Free Prints, Photo Books, Cards, G...",PHOTOGRAPHY,4.6,98716,59.0,5000000,Free,0.0,Everyone,Photography,2018-08-01,5.13.1,5.0 and up,15.424948,11.500002,Top rated app,0.0,2018,5.0,2018-08
2803,FreePrints – Free Photos Delivered,PHOTOGRAPHY,4.8,109500,37.0,1000000,Free,0.0,Everyone,Photography,2018-08-02,2.18.2,4.1 and up,13.815511,11.603680,Top rated app,0.0,2018,4.1,2018-08
2811,"Face Filter, Selfie Editor - Sweet Camera",PHOTOGRAPHY,4.7,142634,22.0,10000000,Free,0.0,Everyone,Photography,2018-07-06,1.5.1,4.4 and up,16.118096,11.868037,Top rated app,0.0,2018,4.4,2018-07
2822,Makeup Editor -Beauty Photo Editor & Selfie Ca...,PHOTOGRAPHY,4.5,3378,30.0,1000000,Free,0.0,Mature 17+,Photography,2018-07-25,1.7,4.1 and up,13.815511,8.125039,Top rated app,0.0,2018,4.1,2018-07
2823,Makeup Photo Editor: Makeup Camera & Makeup Ed...,PHOTOGRAPHY,4.4,10525,25.0,1000000,Free,0.0,Everyone,Photography,2018-07-27,8.9.9,4.0 and up,13.815511,9.261509,Top rated app,0.0,2018,4.0,2018-07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9859,Offline Maps & Navigation,TRAVEL_AND_LOCAL,4.7,193364,33.0,5000000,Free,0.0,Everyone,Travel & Local,2018-06-07,17.4.1,4.0.3 and up,15.424948,12.172330,Top rated app,0.0,2018,4.0,2018-06
9970,Bird - Enjoy The Ride,TRAVEL_AND_LOCAL,4.3,2649,25.0,500000,Free,0.0,Everyone,Travel & Local,2018-08-03,4.1.1,4.4 and up,13.122363,7.881937,Top rated app,0.0,2018,4.4,2018-08
9987,eCooltra: scooter sharing. Share electric scoo...,TRAVEL_AND_LOCAL,4.2,2822,27.0,100000,Free,0.0,Everyone,Travel & Local,2018-07-27,1.21.2,4.1 and up,11.512925,7.945201,Top rated app,0.0,2018,4.1,2018-07
10144,EZ-GUI Ground Station,TOOLS,4.7,3696,29.0,50000,Free,0.0,Everyone,Tools,2017-08-10,4.5,4.3 and up,10.819778,8.215006,Top rated app,0.0,2017,4.3,2017-08


In [52]:
result

,Month,Category,Total_Installs,MoM_Growth,Highlight,Category_Translated
0,2014-11,PHOTOGRAPHY,1000000,NaN,Normal,PHOTOGRAPHY
1,2016-10,PRODUCTIVITY,1000000,NaN,Normal,PRODUCTIVITY
2,2016-12,PERSONALIZATION,2000000,NaN,Normal,PERSONALIZATION
3,2017-03,PHOTOGRAPHY,50000000,49.000000,High Growth,PHOTOGRAPHY
4,2017-06,PHOTOGRAPHY,10000000,-0.800000,Normal,PHOTOGRAPHY
5,2017-07,PHOTOGRAPHY,15000000,0.500000,High Growth,PHOTOGRAPHY
6,2017-08,TOOLS,50000,NaN,Normal,TOOLS
7,2017-09,PERSONALIZATION,1000000,-0.500000,Normal,PERSONALIZATION
8,2017-10,PHOTOGRAPHY,5000000,-0.666667,Normal,PHOTOGRAPHY
9,2017-10,TRAVEL_AND_LOCAL,50000,NaN,Normal,TRAVEL_AND_LOCAL


In [53]:
#Question 5
merged_df['Reviews'] = pd.to_numeric(merged_df['Reviews'], errors='coerce')
merged_df['Rating'] = pd.to_numeric(merged_df['Rating'], errors='coerce')
merged_df['Installs'] = (merged_df['Installs'].astype(str).str.replace('[+,]', '', regex=True))
merged_df['Installs'] = pd.to_numeric(merged_df['Installs'], errors='coerce')
merged_df['Size'] = merged_df['Size'].astype(str)
merged_df['Size'] = merged_df['Size'].str.replace('M','', regex=True)
merged_df['Size'] = merged_df['Size'].str.replace('k','', regex=True)
merged_df['Size'] = pd.to_numeric(merged_df['Size'], errors='coerce')



filtered_df = merged_df[
    (merged_df['Rating'] > 3.5) &
    (merged_df['Reviews'] > 500) &
    (merged_df['Installs'] > 50000) &
    (merged_df['Sentiment_Subjectivity'] > 0.5) &
    (~merged_df['App'].str.contains('S', case=False)) &
    (merged_df['Category'].isin([
        'GAME','BEAUTY','BUSINESS','COMICS',
        'COMMUNICATION','DATING','ENTERTAINMENT',
        'SOCIAL','EVENTS'
    ]))
].copy()



translation = {
    'BEAUTY': 'सौंदर्य',
    'BUSINESS': 'வணிகம்',
    'DATING': 'Dating (Deutsch)'
}

filtered_df['Category_Translated'] = filtered_df['Category'].replace(translation)


TimeWindow = timezone(timedelta(hours=5, minutes=30))
current_time = datetime.now(TimeWindow)



if 17 <= current_time.hour < 19:

    fig15 = px.scatter(
        filtered_df,
        x='Size',
        y='Rating',
        size='Installs',
        color='Category_Translated',
        hover_name='App',
        title='App Size vs Rating (Bubble Size = Installs)',
        size_max=40
    )

    
    fig15.for_each_trace(
        lambda t: t.update(marker=dict(color='pink'))
        if t.name == 'GAME' else None
    )

    fig15.update_layout(
        plot_bgcolor='black',
        paper_bgcolor='black',
        font_color='white',
        title_font={'size':16},
        margin=dict(l=10,r=10,t=30,b=10)
    )

    save_plot_as_html(fig15,"Bubble_Size_Rating.html","Bubble chart showing relationship between app size and rating")

In [54]:
filtered_df

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity,Category_Translated
1790,Google Primer,BUSINESS,4.4,62272,18.0,10000000,Free,0.0,Everyone,Business,"June 26, 2018",3.550.2,4.1 and up,Great,Positive,0.800000,0.750000,வணிகம்
1791,Google Primer,BUSINESS,4.4,62272,18.0,10000000,Free,0.0,Everyone,Business,"June 26, 2018",3.550.2,4.1 and up,Good,Positive,0.700000,0.600000,வணிகம்
1827,Box,BUSINESS,4.2,159872,NaN,10000000,Free,0.0,Everyone,Business,"July 31, 2018",Varies with device,Varies with device,I absolutely love app! It provides fast access...,Positive,0.544940,0.758333,வணிகம்
1830,Box,BUSINESS,4.2,159872,NaN,10000000,Free,0.0,Everyone,Business,"July 31, 2018",Varies with device,Varies with device,A impressive replacement Sharepoint useful peo...,Positive,0.577778,0.611111,வணிகம்
1834,Box,BUSINESS,4.2,159872,NaN,10000000,Free,0.0,Everyone,Business,"July 31, 2018",Varies with device,Varies with device,This incredibly helpful communicating tasks ne...,Positive,0.533333,0.616667,வணிகம்
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59023,Garena Free Fire,GAME,4.5,5534114,53.0,100000000,Free,0.0,Teen,Action,"August 3, 2018",1.21.0,4.0.3 and up,I scratch redemption code purchasing Asus phon...,Positive,0.283333,0.716667,GAME
59024,Garena Free Fire,GAME,4.5,5534114,53.0,100000000,Free,0.0,Teen,Action,"August 3, 2018",1.21.0,4.0.3 and up,Superb game...can u add guns like bazooka rock...,Positive,1.000000,1.000000,GAME
59025,Garena Free Fire,GAME,4.5,5534114,53.0,100000000,Free,0.0,Teen,Action,"August 3, 2018",1.21.0,4.0.3 and up,"More guns must added+more stages,maps scopes g...",Positive,0.900000,0.625000,GAME
59028,Garena Free Fire,GAME,4.5,5534114,53.0,100000000,Free,0.0,Teen,Action,"August 3, 2018",1.21.0,4.0.3 and up,The enemies die take like 30 bullets kill enem...,Neutral,0.000000,0.666667,GAME


In [55]:
# Question 6
merged_df['Reviews'] = pd.to_numeric(merged_df['Reviews'], errors='coerce')

merged_df['Installs'] = (
    merged_df['Installs']
    .astype(str)
    .str.replace('[+,]', '', regex=True)
)

merged_df['Installs'] = pd.to_numeric(merged_df['Installs'], errors='coerce')

merged_df['Last Updated'] = pd.to_datetime(merged_df['Last Updated'], errors='coerce')



filtered_df = merged_df[
    (merged_df['Reviews'] > 500) &
    (merged_df['Category'].str.startswith(('E','C','B'))) &
    (~merged_df['App'].str.startswith(('x','y','z','X','Y','Z'))) &
    (~merged_df['App'].str.contains('S', case=False))
].copy()



filtered_df['Month'] = filtered_df['Last Updated'].dt.to_period('M').astype(str)


result = filtered_df.groupby(['Month','Category']).agg(Total_Installs=('Installs','sum')).reset_index()


result['MoM_Growth'] = result.groupby('Category')['Total_Installs'].pct_change()


result['Growth_Flag'] = result['MoM_Growth'].apply(
    lambda x: 'High Growth' if x > 0.20 else 'Normal'
)


translation = {
    'BEAUTY': 'सौंदर्य',
    'BUSINESS': 'வணிகம்',
    'DATING': 'Dating (Deutsch)'
}

result['Category_Translated'] = result['Category'].replace(translation)


TimeWindow = timezone(timedelta(hours=5, minutes=30))
current_time = datetime.now(TimeWindow)


if 18 <= current_time.hour < 21:

    fig16 = px.line(
        result,
        x='Month',
        y='Total_Installs',
        color='Category_Translated',
        title='Trend of Total Installs Over Time by Category'
    )

    
    high_growth = result[result['MoM_Growth'] > 0.20]

    fig16.add_scatter(
        x=high_growth['Month'],
        y=high_growth['Total_Installs'],
        mode='markers',
        marker=dict(size=10, color='red'),
        name='High Growth (>20%)'
    )

    fig16.update_layout(
        plot_bgcolor='black',
        paper_bgcolor='black',
        font_color='white',
        title_font={'size':16},
        margin=dict(l=10,r=10,t=30,b=10)
    )

    save_plot_as_html(fig16,"Install_Trend_TimeSeries.html","Time series showing installs trend with highlighted growth periods")

In [56]:
filtered_df

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity,Month
1009,Amazon Kindle,BOOKS_AND_REFERENCE,4.2,814080,NaN,100000000,Free,0.0,Teen,Books & Reference,2018-07-27,Varies with device,Varies with device,"OK, despite experience could little intuitive,...",Positive,0.251389,0.516667,2018-07
1010,Amazon Kindle,BOOKS_AND_REFERENCE,4.2,814080,NaN,100000000,Free,0.0,Teen,Books & Reference,2018-07-27,Varies with device,Varies with device,I like reading alternitive Galaxy Tab S2 basic...,Positive,0.079861,0.459722,2018-07
1011,Amazon Kindle,BOOKS_AND_REFERENCE,4.2,814080,NaN,100000000,Free,0.0,Teen,Books & Reference,2018-07-27,Varies with device,Varies with device,I want option remove page flip. Im already irr...,Negative,-0.169141,0.325000,2018-07
1012,Amazon Kindle,BOOKS_AND_REFERENCE,4.2,814080,NaN,100000000,Free,0.0,Teen,Books & Reference,2018-07-27,Varies with device,Varies with device,I used LOVE kindle app. Read s7 edge time. Now...,Negative,-0.020000,0.480000,2018-07
1013,Amazon Kindle,BOOKS_AND_REFERENCE,4.2,814080,NaN,100000000,Free,0.0,Teen,Books & Reference,2018-07-27,Varies with device,Varies with device,The functional ability Kindle library great me...,Positive,0.140000,0.590000,2018-07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56001,Chrome Dev,COMMUNICATION,4.4,63576,NaN,5000000,Free,0.0,Everyone,Communication,2018-08-02,69.0.3497.24,Varies with device,Love it. Use tablet & laptop every day. Can't ...,Positive,0.500000,0.612500,2018-08
56002,Chrome Dev,COMMUNICATION,4.4,63576,NaN,5000000,Free,0.0,Everyone,Communication,2018-08-02,69.0.3497.24,Varies with device,love doesnt even feel like dev super easy,Positive,0.422222,0.700000,2018-08
56003,Chrome Dev,COMMUNICATION,4.4,63576,NaN,5000000,Free,0.0,Everyone,Communication,2018-08-02,69.0.3497.24,Varies with device,"cant open link apps, crashes trying open links",Neutral,0.000000,0.500000,2018-08
56004,Chrome Dev,COMMUNICATION,4.4,63576,NaN,5000000,Free,0.0,Everyone,Communication,2018-08-02,69.0.3497.24,Varies with device,Use full,Positive,0.350000,0.550000,2018-08


In [57]:
result

,Month,Category,Total_Installs,MoM_Growth,Growth_Flag,Category_Translated
0,2014-07,COMMUNICATION,340000000,NaN,Normal,COMMUNICATION
1,2015-01,EDUCATION,3700000,NaN,Normal,EDUCATION
2,2015-06,EDUCATION,35000000,8.459459,High Growth,EDUCATION
3,2015-07,BOOKS_AND_REFERENCE,380000000,NaN,Normal,BOOKS_AND_REFERENCE
4,2015-08,EDUCATION,36000000,0.028571,Normal,EDUCATION
5,2016-06,COMMUNICATION,1000000,-0.997059,Normal,COMMUNICATION
6,2016-09,EDUCATION,1000000,-0.972222,Normal,EDUCATION
7,2016-09,ENTERTAINMENT,240000000,NaN,Normal,ENTERTAINMENT
8,2016-12,EDUCATION,800000,-0.200000,Normal,EDUCATION
9,2017-08,BOOKS_AND_REFERENCE,400000000,0.052632,Normal,BOOKS_AND_REFERENCE


In [58]:
plot_containers_split=plot_containers.split('</div>')

In [59]:
if len(plot_containers_split) > 1:
    final_plot=plot_containers_split[-2]+'</div>'
else:
    final_plot=plot_containers

In [60]:
dashboard_html= """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name=viewport" content="width=device-width,initial-scale-1.0">
    <title> Google Play Store Review Analytics</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
            background-color: #333;
            color: #fff;
            margin: 0;
            padding: 0;
        }}
        .header {{
            display: flex;
            align-items: center;
            justify-content: center;
            padding: 20px;
            background-color: #444
        }}
        .header img {{
            margin: 0 10px;
            height: 50px;
        }}
        .container {{
            display: flex;
            flex-wrap: wrap;
            justify_content: center;
            padding: 20px;
        }}
        .plot-container {{
            border: 2px solid #555
            margin: 10px;
            padding: 10px;
            width: {plot_width}px;
            height: {plot_height}px;
            overflow: hidden;
            position: relative;
            cursor: pointer;
        }}
        .insights {{
            display: none;
            position: absolute;
            right: 10px;
            top: 10px;
            background-color: rgba(0,0,0,0.7);
            padding: 5px;
            border-radius: 5px;
            color: #fff;
        }}
        .plot-container: hover .insights {{
            display: block;
        }}
        </style>
        <script>
            function openPlot(filename) {{
                window.open(filename, '_blank');
                }}
        </script>
    </head>
    <body>
        <div class= "header">
            <img src="https://upload.wikimedia.org/wikipedia/commons/thumb/4/4a/Logo_2013_Google.png/800px-Logo_2013_Google.png" alt="Google Logo">
            <h1>Google Play Store Reviews Analytics</h1>
            <img src="https://upload.wikimedia.org/wikipedia/commons/thumb/7/78/Google_Play_Store_badge_EN.svg/1024px-Google_Play_Store_badge_EN.svg.png" alt="Google Play Store Logo">
        </div>
        <div class="container">
            {plots}
        </div>
    </body>
    </html>
    """


In [61]:
final_html=dashboard_html.format(plots=plot_containers,plot_width=plot_width,plot_height=plot_height)

In [62]:
dashboard_path=os.path.join(html_files_path,"web page.html")

In [63]:
with open(dashboard_path, "w", encoding="utf-8") as f:
    f.write(final_html)

In [64]:
webbrowser.open('file://'+os.path.realpath(dashboard_path))

True